# Standard Kraken2 versus a modern AI approach for metagenomic read classification

## The problem

When a crop shows disease symptoms, one way to identify the cause is to extract all the DNA in the
sample and sequence it without any targeting (**metagenomics**). The sequencer returns many
short reads (here 150 bp) that come from a mixture of organisms, and none of them carries a label.
The first analysis step is **read classification**: deciding, for every read, which organism it
came from, ideally down to the **species** and at least to the **genus**.

This notebook compares two very different ways of doing this:

| | **Kraken2** (standard method) | **AI** (DNA language model, fine-tuned) |
|---|---|---|
| idea | looks up the exact **k-mers** (short DNA words) of each read in a database built from reference genomes | a pretrained **transformer** converts the read into a numeric representation, and a classifier on top predicts the species |
| needs | the reference genomes (the database) | labeled training reads and a GPU for fine-tuning |
| a read it is unsure about | is reported at a **higher rank** (the lowest common ancestor, LCA) or left unclassified | always gets a species name, unless a confidence cut-off is added |

**Question of the exercise:** how accurately does each method assign reads to species and genus,
and how does each one behave on reads from a species it has **never seen**?

## The experiment

1. **Reference panel.** We download from NCBI one genome for each of **18 plant-pathogenic
   bacterial species from 10 genera**: *Xanthomonas, Ralstonia, Pseudomonas, Burkholderia,
   Pantoea, Dickeya, Pectobacterium, Clavibacter, Agrobacterium* and *Xylella*. The panel includes
   the main rice pathogens (*X. oryzae*, *B. glumae*, *P. ananatis*, *D. zeae*).
2. **Simulated reads.** With the read simulator **InSilicoSeq** we generate Illumina-like reads
   from each genome. Because we generated them, we know the true species and genus of every read
   (the **ground truth**), so we can measure accuracy exactly. This is not possible with a real,
   unlabeled sample.
3. **The two classifiers.** Kraken2 gets a database built from the 18 complete genomes. The AI
   model, [Nucleotide Transformer v2 (50M parameters, multi-species)](https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-50m-multi-species),
   is fine-tuned on part of the simulated reads (the training reads).
4. **Part C, known genomes.** Both methods classify the same held-out test reads, which come from
   the 18 panel genomes.
5. **Part D, novel species.** Both methods classify reads from 8 **other species** of the same
   genera, which are neither in the Kraken2 database nor in the AI training data.

## How the notebook is organised

| Section | Content |
|---|---|
| 1 | Setup: GPU check, software installation, NCBI taxonomy |
| 2 | Download of the 18 reference genomes |
| 3 | Read simulation and train / validation / test split |
| 4 | Part A: Kraken2 database and classification |
| 5 | Part B: fine-tuning and evaluation of the AI model |
| 6 | Part C: comparison on held-out reads from known genomes |
| 7 | Part D: comparison on reads from novel species |
| 8 | Summary of the workflow |
| 9 | Structure of your report |

**Before you start:** choose `Runtime > Change runtime type > GPU` (a Colab Pro GPU such as an
A100 is recommended). Run the cells from top to bottom, in order: later cells use variables
created by earlier ones. If the runtime disconnects, start again from the first cell.

# 1. Setup

**Check the GPU.** `nvidia-smi` lists the GPU attached to this Colab session. If you see an error
or no GPU, change the runtime type (see above) before continuing: the AI fine-tuning in Part B
needs a GPU.

In [ ]:
!nvidia-smi

**Install the Python packages.**
- **InSilicoSeq** (`iss`): simulates Illumina reads from genomes.
- **Biopython**: reads and writes FASTA / FASTQ files.
- **transformers, datasets, accelerate** (HuggingFace): load, tokenize and fine-tune the AI model.
- **scikit-learn, seaborn**: metrics (precision, recall, confusion matrix) and plots.

In [ ]:
# Python tools: read simulator + HuggingFace + metrics
!pip install -q InSilicoSeq biopython transformers datasets accelerate scikit-learn seaborn
print("python deps installed")

**Install the command-line tools and download the taxonomy.**
- **Kraken2** (release v2.17.1) is compiled from source into `~/kbin`.
- The **NCBI `datasets`** program is downloaded; it is used to fetch reference genomes.
- The **NCBI taxonomy** is downloaded: `nodes.dmp` says which taxon is the parent of which, and
  at which rank (species, genus, ...); `names.dmp` gives the scientific names. Kraken2 needs the
  taxonomy to build its database, and we use it to translate any taxon ID into its species and
  genus.

Each step is skipped if it was already done in this session. The cell prints the versions and the
taxonomy files.

In [ ]:
%%bash
# --- Kraken2 (stable release v2.17.1, built from source: ~1 min, avoids conda/kernel restart) ---
set -e
if [ ! -x "$HOME/kbin/kraken2" ]; then
  rm -rf /content/kraken2_src
  git clone -q --depth 1 --branch v2.17.1 https://github.com/DerrickWood/kraken2 /content/kraken2_src
  cd /content/kraken2_src
  ./install_kraken2.sh "$HOME/kbin" >/dev/null 2>&1
fi
"$HOME/kbin/kraken2" --version | head -1

# --- NCBI datasets CLI (single static binary) for fetching reference genomes ---
if [ ! -x /content/datasets ]; then
  curl -sL 'https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets' -o /content/datasets
  chmod +x /content/datasets
fi
/content/datasets --version

# --- NCBI taxonomy (names/nodes only): Kraken2 DB + resolving species / genus ---
mkdir -p /content/taxonomy
if [ ! -f /content/taxonomy/nodes.dmp ]; then
  wget -q https://ftp.ncbi.nlm.nih.gov/pub/taxonomy/taxdump.tar.gz -O /content/taxdump.tar.gz
  tar -xzf /content/taxdump.tar.gz -C /content/taxonomy names.dmp nodes.dmp
fi
ls -la /content/taxonomy

**Python helpers.** This cell
- adds the new programs to the `PATH` and fixes the random seeds (reproducible results);
- defines `run(cmd)`, which runs a command-line program and shows its error message if it fails;
- loads the NCBI taxonomy into memory (`PARENT`, `RANK`, `NAME`);
- defines `ancestor(taxid, rank)`, which walks a taxon ID (for example a strain) up the tree until
  it reaches the requested rank, for example `"species"` or `"genus"`.

It prints the number of taxa loaded and the number of CPUs available.

In [ ]:
import os, json
os.environ["PATH"] = os.path.expanduser("~/kbin") + ":/content:" + os.environ["PATH"]
import glob, random, shutil, subprocess, time
from pathlib import Path
import numpy as np, pandas as pd
from Bio import SeqIO
random.seed(0); np.random.seed(0)
NCPU = os.cpu_count() or 2

def run(cmd):
    """Run a command; on failure show its output instead of a bare CalledProcessError."""
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError("command failed: " + " ".join(cmd[:4]) + " ...")
    return r.stdout

# NCBI taxonomy in memory, so any taxid can be walked up to its species / genus
PARENT, RANK, NAME = {}, {}, {}
with open("/content/taxonomy/nodes.dmp") as f:
    for line in f:
        t, p, r = line.split("\t|\t", 3)[:3]
        PARENT[t], RANK[t] = p, r
with open("/content/taxonomy/names.dmp") as f:
    for line in f:
        if line.endswith("scientific name\t|\n"):
            t, n = line.split("\t|\t", 2)[:2]
            NAME[t] = n

def ancestor(taxid, rank):
    """taxid of the ancestor at `rank` (e.g. 'species', 'genus'), or None."""
    t = str(taxid)
    while t in PARENT:
        if RANK[t] == rank:
            return t
        if PARENT[t] == t:
            break
        t = PARENT[t]
    return None

print(f"taxonomy: {len(PARENT):,} taxa | CPUs: {NCPU}")
print("kraken2:", shutil.which("kraken2"), "| datasets:", shutil.which("datasets"))

# 2. Download the reference panel (18 plant-pathogen species)

`PANEL` lists the species to download, grouped by genus. For each species the cell asks NCBI for
its **reference genome** (if there is none, a complete RefSeq genome), downloads it and checks it
against the taxonomy:
- the species and genus labels are taken from the **NCBI taxonomy**, so if NCBI files a genome
  under a different species than requested, the label follows NCBI;
- a genome is skipped if it is not in the expected genus or if its species is already in the
  panel.

For every genome the cell prints the species, the **NCBI accession** (record these for your
report) and the genome size. At the end it builds the lists used everywhere else: `SPECIES`
(the AI class order), `GENERA`, and dictionaries that translate taxon IDs to names.

These genomes are used twice: they make up the **Kraken2 database**, and they are the source of
the **training and test reads** for the AI.

In [ ]:
def ncbi_hits(*args):
    r = subprocess.run(["datasets", "summary", "genome", *args, "--as-json-lines"],
                       capture_output=True, text=True)
    hits = [json.loads(l) for l in r.stdout.splitlines() if l.startswith("{")]
    return sorted(hits, key=lambda h: (not h["accession"].startswith("GCF"), h["accession"]))

def fetch_genome(query, outdir, by="taxon"):
    """-> (accession, taxid, fna) or None. By taxon: reference genome, else complete RefSeq."""
    if by == "taxon":
        hits = (ncbi_hits("taxon", query, "--reference") or
                ncbi_hits("taxon", query, "--assembly-level", "complete",
                          "--assembly-source", "RefSeq", "--limit", "5"))
    else:
        hits = ncbi_hits("accession", query)
    if not hits:
        return None
    acc = hits[0]["accession"]
    org = hits[0]["organism"]
    taxid = str(org.get("tax_id", org.get("taxId")))
    if not glob.glob(f"{outdir}/ncbi_dataset/data/{acc}/*.fna"):
        run(["datasets", "download", "genome", "accession", acc,
             "--include", "genome", "--filename", f"{outdir}.zip"])
        run(["unzip", "-qo", f"{outdir}.zip", "-d", outdir])
    return acc, taxid, sorted(glob.glob(f"{outdir}/ncbi_dataset/data/{acc}/*.fna"))[0]

def resolve(query, genus, outdir, exclude=()):
    """Fetch a genome for `query`; check it really is in `genus` and not an excluded species."""
    got = fetch_genome(query, outdir)
    if got is None:
        print(f"  !! {query}: no genome on NCBI — skipped"); return None
    acc, taxid, fna = got
    sp_t, gn_t = ancestor(taxid, "species"), ancestor(taxid, "genus")
    species = NAME.get(sp_t)
    if NAME.get(gn_t) != genus or species is None or species in exclude:
        print(f"  !! {query}: {acc} is {species} ({NAME.get(gn_t)}) — skipped"); return None
    size = sum(len(l) - 1 for l in open(fna) if not l.startswith(">"))
    print(f"  {species:32s} {acc:18s} {size/1e6:5.2f} Mb")
    return dict(genus=genus, acc=acc, taxid=taxid, sp_taxid=sp_t, gn_taxid=gn_t, fna=fna)

PANEL = {   # genus -> species to fetch (all plant pathogens)
    "Xanthomonas":    ["Xanthomonas oryzae", "Xanthomonas campestris"],
    "Ralstonia":      ["Ralstonia solanacearum", "Ralstonia pseudosolanacearum"],
    "Pseudomonas":    ["Pseudomonas syringae", "Pseudomonas savastanoi"],
    "Burkholderia":   ["Burkholderia glumae", "Burkholderia gladioli"],
    "Pantoea":        ["Pantoea ananatis", "Pantoea stewartii"],
    "Dickeya":        ["Dickeya zeae", "Dickeya dadantii"],
    "Pectobacterium": ["Pectobacterium carotovorum", "Pectobacterium atrosepticum"],
    "Clavibacter":    ["Clavibacter michiganensis", "Clavibacter sepedonicus"],
    "Agrobacterium":  ["Agrobacterium tumefaciens"],
    "Xylella":        ["Xylella fastidiosa"],
}
REFS = {}   # species label (NCBI name) -> genome info
os.makedirs("/content/genomes", exist_ok=True)
for genus, queries in PANEL.items():
    for q in queries:
        info = resolve(q, genus, f"/content/genomes/{q.replace(' ', '_')}", exclude=REFS)
        if info:
            REFS[NAME[info["sp_taxid"]]] = info

SPECIES = sorted(REFS)                                   # AI class order
GENERA  = sorted({r["genus"] for r in REFS.values()})
SP2IDX  = {s: i for i, s in enumerate(SPECIES)}
SP2GEN  = {s: REFS[s]["genus"] for s in SPECIES}
SPTAX2NAME = {REFS[s]["sp_taxid"]: s for s in SPECIES}
GNTAX2NAME = {REFS[s]["gn_taxid"]: REFS[s]["genus"] for s in SPECIES}
print(f"\npanel: {len(SPECIES)} species in {len(GENERA)} genera")

# 3. Simulate reads with known ground truth

`simulate()` uses **InSilicoSeq** with its NovaSeq error model to generate **150-bp Illumina
reads** from one genome:
- contigs shorter than 2 kb are removed, and the remaining contigs are sampled in proportion to
  their length (InSilicoSeq's default would over-sample small plasmids);
- InSilicoSeq produces read pairs; only read 1 of each pair is kept, so the reads are treated as
  single-end;
- reads shorter than 100 bp or containing `N` are discarded.

`READS_PER_SPECIES` reads are simulated per species. Every read is labeled with its species and
genus, then all reads are pooled, shuffled and split:

| split | share | used for |
|---|---|---|
| **train** | 70 % | the AI learns from these reads |
| **val** | 15 % | checks the AI after each epoch (it never trains on them) |
| **test** | 15 % | final, held-out evaluation of **both** tools |

The cell prints the number of reads per species in each split. The test reads are stored in
`test_df`.

**Keep in mind:** Kraken2's database will contain the **complete genomes**, whereas the AI only
learns from the **training reads**. How much of each genome does each tool get to see?

In [ ]:
def simulate(fna, n_reads, prefix, seed=0):
    """Simulate n_reads single-end reads (R1 of NovaSeq pairs); drop reads with N or < 100 bp."""
    recs = [r for r in SeqIO.parse(fna, "fasta") if len(r.seq) >= 2000]
    tot = sum(len(r.seq) for r in recs)
    SeqIO.write(recs, f"{prefix}.fna", "fasta")
    with open(f"{prefix}.abund", "w") as f:
        for r in recs:
            f.write(f"{r.id}\t{len(r.seq) / tot}\n")
    run(["iss", "generate", "--genomes", f"{prefix}.fna", "--abundance_file", f"{prefix}.abund",
         "--n_reads", str(2 * n_reads),          # iss counts both mates of a pair
         "--model", "NovaSeq", "--seed", str(seed), "--cpus", str(NCPU), "--output", prefix])
    seqs = (str(r.seq).upper() for r in SeqIO.parse(f"{prefix}_R1.fastq", "fastq"))
    return [s for s in seqs if len(s) >= 100 and set(s) <= set("ACGT")]

READS_PER_SPECIES = 1500      # keep small so Colab runs fast; raise for a better AI model
rows = []
for s in SPECIES:
    for seq in simulate(REFS[s]["fna"], READS_PER_SPECIES, f"/content/sim_{s.replace(' ', '_')}"):
        rows.append({"seq": seq, "species": s, "genus": SP2GEN[s]})

df = pd.DataFrame(rows).sample(frac=1, random_state=0).reset_index(drop=True)
df["read_id"] = [f"r{i}" for i in range(len(df))]          # clean unique ids
n = len(df); a, b = int(0.70*n), int(0.85*n)
df["split"] = ["train"]*a + ["val"]*(b-a) + ["test"]*(n-b)
print("reads:", n, "| per split:", df["split"].value_counts().to_dict())
print(df.groupby(["species", "split"]).size().unstack(fill_value=0))
test_df = df[df.split == "test"].reset_index(drop=True)

# 4. Part A: standard Kraken2

### Build the Kraken2 database

The cell builds a **custom Kraken2 database** in `/content/k2db` from the 18 panel genomes:
1. the NCBI taxonomy files are linked into the database folder;
2. every contig header is tagged with its genome's NCBI taxon ID (`kraken:taxid|...`), so that
   Kraken2 knows which taxon each sequence belongs to and can call reads down to species (or
   strain);
3. the tagged genomes are added to the library (`kraken2-build --add-to-library`) and the
   database is built (`kraken2-build --build`).

This takes a few minutes. The cell prints a message when the database is ready.

In [ ]:
DB = "/content/k2db"
shutil.rmtree(DB, ignore_errors=True)
os.makedirs(f"{DB}/taxonomy")
for f in ("names.dmp", "nodes.dmp"):
    os.symlink(f"/content/taxonomy/{f}", f"{DB}/taxonomy/{f}")

for s in SPECIES:   # tag headers with the genome's taxid, add to the Kraken2 library
    r = REFS[s]
    tagged = f"/content/{s.replace(' ', '_')}.tagged.fna"
    with open(r["fna"]) as fi, open(tagged, "w") as fo:
        for line in fi:
            if line.startswith(">"):
                fo.write(f">{line[1:].split()[0]}|kraken:taxid|{r['taxid']}\n")
            else:
                fo.write(line)
    run(["kraken2-build", "--add-to-library", tagged, "--db", DB, "--no-masking"])
print("building DB (hash)...")
run(["kraken2-build", "--build", "--db", DB, "--threads", str(NCPU)])
print("Kraken2 DB built:", len(SPECIES), "genomes")

### Classify the test reads with Kraken2

`run_kraken()` writes the reads to a FASTA file, runs `kraken2` on it and reads the per-read
output: `C`/`U` (classified / unclassified), the read ID and the assigned taxon ID.
`kraken_call()` then walks each taxon ID up the taxonomy to a **species call** and a
**genus call**:

| Kraken2 result | species call | genus call |
|---|---|---|
| a panel species (or one of its strains) | that species | its genus |
| only the genus (read shared by two species of the same genus) | `above_species` | the genus |
| above genus (for example a conserved gene shared across genera) | `above_genus` | `above_genus` |
| no k-mer hits | `unclassified` | `unclassified` |

`K2_CONFIDENCE` is Kraken2's `--confidence` option: the fraction of a read's k-mers that must
support a taxon before the read is assigned to it (0 = Kraken2's default, the most permissive).

The calls are added to `test_df` as columns `k2_species` and `k2_genus`. The cell prints the run
time and the number of test reads called per genus.

In [ ]:
K2_CONFIDENCE = 0.0    # Kraken2 --confidence (0 = default); try 0.1 in the exercises

def kraken_call(c, taxid):
    """Kraken2 output line -> (species call, genus call), see table above."""
    if c == "U":
        return "unclassified", "unclassified"
    genus = GNTAX2NAME.get(ancestor(taxid, "genus"), "above_genus")
    if genus == "above_genus":
        return "above_genus", "above_genus"
    return SPTAX2NAME.get(ancestor(taxid, "species"), "above_species"), genus

def run_kraken(d, name):
    """Classify the reads of DataFrame d; adds k2_species / k2_genus; returns seconds."""
    fa, out = f"/content/{name}.fa", f"/content/{name}.k2"
    with open(fa, "w") as f:
        for r in d.itertuples():
            f.write(f">{r.read_id}\n{r.seq}\n")
    t0 = time.time()
    run(["kraken2", "--db", DB, "--threads", str(NCPU), "--confidence", str(K2_CONFIDENCE),
         "--output", out, "--report", f"/content/{name}.k2report", fa])
    dt = time.time() - t0
    calls = {}
    for line in open(out):
        c, rid, taxid = line.split("\t")[:3]
        calls[rid] = kraken_call(c, taxid)
    d["k2_species"] = [calls[r][0] for r in d.read_id]
    d["k2_genus"]   = [calls[r][1] for r in d.read_id]
    return dt

TIMES = {"Kraken2": run_kraken(test_df, "test")}
print(f"Kraken2 classified {len(test_df)} reads in {TIMES['Kraken2']:.1f}s")
print(test_df["k2_genus"].value_counts())

# 5. Part B: the AI approach, fine-tuning a DNA language model

### The model: Nucleotide Transformer v2, 50M parameters, multi-species
`MODEL_NAME = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"` is a pretrained DNA
transformer (about 50 million parameters). Pretraining taught it general features of DNA
sequences, but it knows nothing about our 18 species yet. Its tokenizer cuts DNA into **6-mers**,
so a 150-bp read becomes 25 tokens plus a start token `<cls>`.

### Our strategy: full fine-tuning as an 18-species read classifier
1. **New classification head.** The model is loaded with a new, randomly initialised output layer
   with **18 outputs, one per species**.
2. **Full fine-tuning.** *All* weights (transformer and head) are updated on the labeled
   training reads.
3. **Training settings.** Learning rate 1e-4 (decaying linearly to 0), batch size 64, **20
   epochs**, fp32 precision. The model after the **last** epoch is used.
4. **From species to genus.** The model predicts only species; the **genus probability** of a
   read is the sum of the probabilities of that genus's species.
5. **Confidence cut-off (`AI_MIN_CONF`).** A plain classifier *always* names a species, even for
   DNA it has never seen. With the cut-off it may abstain, similar to Kraken2: if the species
   probability is below the cut-off the species call becomes `above_species`; if the genus
   probability is also below it, the read is `unclassified`. The AI is reported **with and
   without** the cut-off.

### Load the model

The first part of the cell applies two **compatibility patches**. The Nucleotide Transformer code
stored on the HuggingFace Hub was written for an older version of `transformers` and uses helpers
that newer versions removed; the patches add them back. They do not change the model.

The second part loads the tokenizer and the model, with a new head of `len(SPECIES)` outputs.
The load report it prints shows `lm_head.* UNEXPECTED` (the pretraining head, which is discarded)
and `classifier.* MISSING` (the new head, which will be trained). This is expected.

In [ ]:
import transformers
import transformers.pytorch_utils as pt_utils
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification, PreTrainedModel
import torch

# --- compatibility patches for the model code on the HuggingFace Hub (written for older transformers) ---
# 1) newer transformers expect this attribute on every model class
PreTrainedModel.all_tied_weights_keys = {}

# 2) newer transformers removed this helper, which the model code imports
if not hasattr(pt_utils, "find_pruneable_heads_and_indices"):
    def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
        if len(heads) == 0:
            return heads, None
        mask = torch.ones(n_heads, head_size)
        heads = set(heads) - already_pruned_heads
        for head in heads:
            mask[head] = 0
        mask = mask.view(-1).eq(1)
        index = torch.arange(len(mask))[mask].long()
        return heads, index
    pt_utils.find_pruneable_heads_and_indices = find_pruneable_heads_and_indices

# --- load the pretrained model with a new 18-species classification head ---
MODEL_NAME = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"
device = "cuda" if torch.cuda.is_available() else "cpu"

tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

cfg = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
cfg.num_labels = len(SPECIES)
cfg.id2label = {i: s for i, s in enumerate(SPECIES)}
cfg.label2id = {s: i for i, s in enumerate(SPECIES)}
cfg.add_cross_attention = False
cfg.is_decoder = False

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, config=cfg, trust_remote_code=True, low_cpu_mem_usage=False
).to(device)
print("model loaded on", device, "|", sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")

### Tokenize the reads

Each read is converted to token IDs and padded to the same length (`MAXLEN` tokens); padded
positions are masked so the model ignores them. The label of each read is its species index
(0 to 17, in the order of `SPECIES`). The cell checks that no read is longer than `MAXLEN`
(no read is cut), and prints the number of reads per split and the longest read in tokens.

In [ ]:
from datasets import Dataset

def to_ds(sub):
    return Dataset.from_dict({"seq": sub.seq.tolist(),
                              "label": [SP2IDX[s] for s in sub.species]})
ds = {s: to_ds(df[df.split == s]) for s in ("train", "val", "test")}

MAXLEN = 32           # max TOKENS; a 150-bp read is 26 tokens (25 6-mers + <cls>); padding is masked
def tok_fn(b):
    enc = tok(b["seq"], truncation=True, padding="max_length", max_length=MAXLEN)
    enc.pop("token_type_ids", None)
    return enc
ds = {s: d.map(tok_fn, batched=True, remove_columns=["seq"]) for s, d in ds.items()}
longest = max(max(sum(m) for m in d["attention_mask"]) for d in ds.values())
assert longest < MAXLEN, "some reads were truncated: raise MAXLEN"
print({s: len(d) for s, d in ds.items()}, "| longest read:", longest, "tokens")

### Fine-tune the model

The first part of the cell adds two more compatibility helpers (attention-mask and head-mask
functions) that the model code expects. The second part sets up the HuggingFace `Trainer` with the
settings described above and trains the model. After every epoch the Trainer shows the training
loss, the validation loss and the species accuracy on the **validation** reads.

This is the longest step of the notebook (the cell prints how many minutes it took).

In [ ]:
from transformers import TrainingArguments, Trainer
import types

# --- two more compatibility helpers that the model code expects (removed in newer transformers) ---
def custom_get_extended_attention_mask(self, attention_mask, input_shape, device=None, dtype=None):
    if attention_mask.dim() == 3:
        extended_attention_mask = attention_mask[:, None, :, :]
    elif attention_mask.dim() == 2:
        extended_attention_mask = attention_mask[:, None, None, :]
    else:
        raise ValueError("Wrong shape for input_ids or attention_mask")
    extended_attention_mask = extended_attention_mask.to(dtype=torch.float32)
    extended_attention_mask = (1.0 - extended_attention_mask) * torch.finfo(torch.float32).min
    return extended_attention_mask

def custom_get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
    if head_mask is not None:
        return head_mask
    return [None] * num_hidden_layers

for m in [model] + ([model.esm] if hasattr(model, "esm") else []):
    if not hasattr(m, "get_extended_attention_mask"):
        m.get_extended_attention_mask = types.MethodType(custom_get_extended_attention_mask, m)
    if not hasattr(m, "get_head_mask"):
        m.get_head_mask = types.MethodType(custom_get_head_mask, m)

# --- fine-tuning ---
def compute_metrics(p):
    logits = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    return {"accuracy": float((np.argmax(logits, axis=1) == p.label_ids).mean())}

args = TrainingArguments(
    output_dir="/content/ai_out",
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    num_train_epochs=20,       # 20 passes over the training reads
    learning_rate=1e-4,        # AdamW peak learning rate, decays linearly to 0
    eval_strategy="epoch",     # evaluate on the validation reads after every epoch
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    fp16=False,                # full fp32 precision (simplest and most stable)
)
trainer = Trainer(model=model, args=args, train_dataset=ds["train"],
                  eval_dataset=ds["val"], compute_metrics=compute_metrics)
t0 = time.time(); trainer.train(); ai_train_time = time.time() - t0
print(f"fine-tuning done in {ai_train_time/60:.1f} min (val accuracy = species level)")

### Did the model learn, or memorize?

The final model is evaluated on the reads it **trained on** and on the **validation** reads it has
never seen. The table shows the loss and the species accuracy on each set. Chance level for 18
species is 1/18, about **0.056**.

Hint: compare training and validation accuracy. A model that has learned general features
should score similarly on both.

In [ ]:
# evaluate the final model on the training reads and on the validation reads
print("Evaluating on training set...")
train_results = trainer.evaluate(eval_dataset=ds["train"])

print("Evaluating on validation set...")
val_results = trainer.evaluate(eval_dataset=ds["val"])

comparison_df = pd.DataFrame({
    "Metric": ["Loss", "Accuracy"],
    "Training Set": [train_results["eval_loss"], train_results["eval_accuracy"]],
    "Validation Set": [val_results["eval_loss"], val_results["eval_accuracy"]]
})
display(comparison_df)

### Learning curves

Left: training loss (recorded every 50 steps) and validation loss (after every epoch). Right:
validation accuracy after every epoch, with the final training accuracy as a single point.

Hint: follow the validation loss epoch by epoch. When is it lowest, and what happens afterwards?

In [ ]:
import matplotlib.pyplot as plt

# training loss and validation metrics recorded by the Trainer
history = trainer.state.log_history

train_loss = []
train_loss_epochs = []
val_loss = []
val_loss_epochs = []
val_accuracy = []
val_accuracy_epochs = []

for log in history:
    if 'epoch' in log:
        ep = log['epoch']
        if 'loss' in log:
            train_loss.append(log['loss'])
            train_loss_epochs.append(ep)
        if 'eval_loss' in log:
            val_loss.append(log['eval_loss'])
            val_loss_epochs.append(ep)
        if 'eval_accuracy' in log:
            val_accuracy.append(log['eval_accuracy'])
            val_accuracy_epochs.append(ep)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Side-by-side comparison: Loss curves
ax1.plot(train_loss_epochs, train_loss, 'tab:red', label='Training Loss', alpha=0.7)
ax1.plot(val_loss_epochs, val_loss, 'tab:blue', label='Validation Loss', linestyle='--', marker='o')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.set_title('Training vs. Validation Loss')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend()

# Side-by-side comparison: Accuracy curves
# training accuracy was measured only on the final model; validation accuracy after every epoch
ax2.plot(val_accuracy_epochs, val_accuracy, 'tab:blue', label='Validation Accuracy', marker='s')
if len(val_accuracy_epochs) > 0:
    final_epoch = val_accuracy_epochs[-1]
    ax2.plot(final_epoch, comparison_df.loc[1, 'Training Set'], 'ro', markersize=10, label=f'Final Training Accuracy ({comparison_df.loc[1, "Training Set"]:.3f})')

ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Accuracy Evolution')
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend()

plt.tight_layout()
plt.show()

### Predict the test reads with the AI

`ai_predict()` tokenizes the reads, runs the fine-tuned model on them and turns its outputs into
**species probabilities** (softmax over the 18 outputs) and **genus probabilities** (species
probabilities summed per genus, using the membership matrix `G`). It adds two sets of columns to
the table:
- `ai_species`, `ai_genus`: always the most probable class;
- `aiT_species`, `aiT_genus`: with the `AI_MIN_CONF` cut-off, so the model may abstain
  (`above_species` or `unclassified`).

The cell prints the prediction time and the number of test reads called per genus.

In [ ]:
AI_MIN_CONF = 0.90     # confidence cut-off for the abstaining AI ("aiT_*" columns)

SPECIES_A, GENERA_A = np.array(SPECIES), np.array(GENERA)
G = np.zeros((len(SPECIES), len(GENERA)))          # species -> genus membership
for i, s in enumerate(SPECIES):
    G[i, GENERA.index(SP2GEN[s])] = 1

def ai_predict(d):
    """Classify the reads of DataFrame d; adds ai_* (always calls) and aiT_* (with cut-off)."""
    dset = Dataset.from_dict({"seq": d.seq.tolist()}).map(tok_fn, batched=True, remove_columns=["seq"])
    t0 = time.time()
    lg = trainer.predict(dset).predictions
    dt = time.time() - t0
    lg = lg[0] if isinstance(lg, tuple) else lg
    p = torch.softmax(torch.tensor(lg).float(), dim=-1).numpy()   # species probabilities
    gp = p @ G                                                     # genus probabilities
    d["ai_species"] = SPECIES_A[p.argmax(1)]
    d["ai_genus"]   = GENERA_A[gp.argmax(1)]
    gen_ok, sp_ok = gp.max(1) >= AI_MIN_CONF, p.max(1) >= AI_MIN_CONF
    d["aiT_genus"]   = np.where(gen_ok, d["ai_genus"], "unclassified")
    d["aiT_species"] = np.where(sp_ok, d["ai_species"], np.where(gen_ok, "above_species", "unclassified"))
    return dt

TIMES["AI (transformer)"] = ai_predict(test_df)
print(f"AI classified {len(test_df)} test reads in {TIMES['AI (transformer)']:.1f}s")
print(test_df["ai_genus"].value_counts())

# 6. Part C: head-to-head on held-out reads (genomes in the database / training set)

Both methods are scored against the **known ground truth** on the same test reads, at **genus**
and **species** level. This cell defines the scoring functions and prints a scoreboard for each
rank with three numbers per tool:

- **accuracy**: correct calls / all reads (a no-call counts as wrong);
- **precision_of_calls**: correct calls / reads that received a call (how far you can trust a
  call);
- **called_%**: percentage of reads that received a call at this rank.

`TOOLS` lists the three versions compared: Kraken2, the AI, and the AI with the confidence
cut-off. It also defines `plot_cm()`, used two cells below to draw confusion matrices.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns

TOOLS  = {"Kraken2": "k2", "AI": "ai", f"AI (conf ≥ {AI_MIN_CONF})": "aiT"}
NOCALL = ["unclassified", "above_species", "above_genus"]

def scoreboard(d, rank):
    rows = {}
    for name, p in TOOLS.items():
        pred, truth = d[f"{p}_{rank}"], d[rank]
        called = ~pred.isin(NOCALL)
        rows[name] = {"accuracy": (pred == truth).mean(),
                      "precision_of_calls": (pred[called] == truth[called]).mean() if called.any() else np.nan,
                      "called_%": 100 * called.mean()}
    return pd.DataFrame(rows).T.round(3)

def plot_cm(d, rank, title):
    true_labs = SPECIES if rank == "species" else GENERA
    fig, axes = plt.subplots(1, 2, figsize=(24, 10) if rank == "species" else (18, 7))
    for ax, (name, p) in zip(axes, [("Kraken2", "k2"), ("AI", "ai")]):
        pred = d[f"{p}_{rank}"]
        labs = list(true_labs) + [x for x in NOCALL if (pred == x).any()]
        cm = confusion_matrix(d[rank], pred, labels=labs)[:len(true_labs)]
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                    xticklabels=labs, yticklabels=true_labs, ax=ax)
        ax.set_title(f"{name} — {rank} ({title}), acc={(pred == d[rank]).mean():.3f}")
        ax.set_xlabel("predicted"); ax.set_ylabel("true")
    plt.tight_layout(); plt.show()

for rank in ["genus", "species"]:
    print(f"\n######## {rank.upper()} level ########")
    print(scoreboard(test_df, rank))

**Per-species precision and recall.** For each species, `classification_report` gives the
precision (of the reads called as this species, how many really are), the recall (of the reads of
this species, how many were called correctly), the F1-score (their harmonic mean) and the support
(number of test reads). Kraken2 and the AI (without cut-off) are shown one after the other.

In [ ]:
# per-species precision / recall (Kraken2 vs AI)
for name, p in [("Kraken2", "k2"), ("AI", "ai")]:
    pred = test_df[f"{p}_species"]
    print(f"\n===== {name} — species level =====")
    print(classification_report(test_df.species, pred,
          labels=SPECIES + [x for x in NOCALL if (pred == x).any()], zero_division=0))

**Confusion matrices.** Rows are the true genus (or species), columns the predicted one; the
numbers are read counts. A perfect classifier puts all reads on the diagonal. The extra columns on
the right (`above_species`, `above_genus`, `unclassified`) show the reads that were not called at
that rank. Kraken2 is on the left, the AI on the right; the title gives the accuracy.

In [ ]:
plot_cm(test_df, "genus", "held-out reads")
plot_cm(test_df, "species", "held-out reads")

**Speed.** Time each tool needed to classify the test reads. Kraken2's time includes loading its
database and runs on the CPU; the AI's time is prediction only, on the GPU, and does not include
fine-tuning, which is printed separately.

In [ ]:
# speed (inference only; Kraken2 includes DB loading on CPU, the AI excludes GPU training)
print(pd.Series(TIMES, name="inference_s").round(1))
print(f"AI fine-tuning: {ai_train_time/60:.1f} min")

### What to look at in Part C

- **Kraken2:** why could it score so well here? Think about where the test reads came from.
- **AI:** compare its accuracy with chance level (10 % for genus, 5.6 % for species) and with
  Kraken2.
- **Genus confusion matrix:** which genera does the AI mix up? Are they related?
- **Cut-off:** with `AI_MIN_CONF`, how many reads are still called, and how precise are they?
- **Speed:** compare the inference times, and remember the fine-tuning time.

# 7. Part D: the fair test, novel species not in the database

Part C graded both tools on genomes they had already seen: Kraken2 has their exact k-mers, and
the AI was trained on reads from them. The more realistic question is **what happens with a
genome that neither tool has seen?**

`NOVEL` lists a **different species for 8 of the genera**. The cell downloads their genomes with
the same checks as in Section 2 (they must be in the expected genus and must not be a panel
species), simulates `NOVEL_READS` reads per species, and classifies them with **both** tools,
with the **same** Kraken2 database (no rebuild) and the **same** AI model (no retraining). The
calls are stored in `novel_df`.

How to score these reads:
- **Genus level:** the correct answer is the genus.
- **Species level:** the correct answer is "none of my species". Any panel species name is a
  **false species call**.

In [ ]:
NOVEL = {   # a DIFFERENT species per genus -> not in the DB / not seen in training
    "Xanthomonas":    "Xanthomonas euvesicatoria",
    "Ralstonia":      "Ralstonia syzygii",
    "Pseudomonas":    "Pseudomonas fluorescens",
    "Burkholderia":   "Burkholderia plantarii",
    "Pantoea":        "Pantoea agglomerans",
    "Dickeya":        "Dickeya solani",
    "Pectobacterium": "Pectobacterium brasiliense",
    "Clavibacter":    "Clavibacter nebraskensis",
}
NOVEL_READS = 800
os.makedirs("/content/novel", exist_ok=True)
nrows = []
for genus, q in NOVEL.items():
    if genus not in GENERA:
        continue
    info = resolve(q, genus, f"/content/novel/{q.replace(' ', '_')}", exclude=REFS)
    if not info:
        continue
    sp = NAME[info["sp_taxid"]]
    for seq in simulate(info["fna"], NOVEL_READS, f"/content/novelsim_{genus}", seed=3):
        nrows.append({"read_id": f"n{len(nrows)}", "seq": seq, "species": sp, "genus": genus})
novel_df = pd.DataFrame(nrows)
print("novel reads:", len(novel_df), "|", novel_df.species.value_counts().to_dict())

run_kraken(novel_df, "novel")     # SAME DB, no rebuild
ai_predict(novel_df)              # SAME model, no retraining
print("classified novel reads with both tools.")

**Score the novel reads.** The cell prints, in order:
1. the genus-level scoreboard for the novel reads and the genus confusion matrices;
2. the percentage of **false species calls** per tool (reads that received the name of a panel
   species);
3. a bar plot and a table comparing genus accuracy on known genomes (Part C) and on novel
   species (Part D);
4. the percentage of novel reads that Kraken2 left unclassified or placed above genus.

In [ ]:
print("######## NOVEL species — GENUS level ########")
print(scoreboard(novel_df, "genus"))
plot_cm(novel_df, "genus", "novel species")

# species level: every species name is wrong here (the true species is not in the panel)
false_sp = pd.Series({name: 100 * novel_df[f"{p}_species"].isin(SPECIES).mean()
                      for name, p in TOOLS.items()}, name="false species calls %")
print("\n######## NOVEL species — SPECIES level ########")
print(false_sp.round(1))

# known vs novel genus accuracy, side by side
comp = pd.DataFrame({name: [(test_df[f"{p}_genus"] == test_df.genus).mean(),
                            (novel_df[f"{p}_genus"] == novel_df.genus).mean()]
                     for name, p in TOOLS.items()},
                    index=["known genome\n(in DB)", "novel species\n(not in DB)"])
ax = comp.plot(kind="bar", figsize=(8, 4), rot=0)
ax.set_ylabel("genus accuracy"); ax.set_title("Known genome vs novel species")
for cont in ax.containers: ax.bar_label(cont, fmt="%.2f")
plt.tight_layout(); plt.show()
print(comp.round(3))
print(f"\nKraken2 on novel reads: {100*(novel_df.k2_genus == 'unclassified').mean():.1f}% unclassified, "
      f"{100*(novel_df.k2_genus == 'above_genus').mean():.1f}% above genus")

### What to look at in Part D

- **Kraken2, genus level:** how much does accuracy drop compared with Part C? Compare accuracy,
  precision of calls and percentage unclassified.
- **Kraken2, species level:** how often does it name a panel species for a read from a novel
  species? Look again at the table in Part A on how Kraken2 reports reads at a higher rank.
- **AI:** compare its genus accuracy on known and novel reads. Is a similar score a sign of
  generalization? Relate it to what you saw in the learning curves.
- **Cut-off:** how many false species calls remain with `AI_MIN_CONF`?

# 8. Summary of the workflow

In this notebook you:
- downloaded 18 plant-pathogen reference genomes from NCBI and simulated labeled 150-bp reads
  from them (train / validation / test split);
- built a custom Kraken2 database from the 18 genomes and translated every Kraken2 call into a
  species and a genus call with the NCBI taxonomy;
- fine-tuned the Nucleotide Transformer v2 (50M) as an 18-species read classifier (genus =
  summed species probabilities), with and without a confidence cut-off, and checked it for
  overfitting;
- **Part C:** scored both tools on held-out reads from the panel genomes (accuracy, precision of
  calls, confusion matrices, speed);
- **Part D:** scored both tools on reads from novel species that are neither in the database nor
  in the training data.

# 9. Your report: structure and content

Write a short report (about 2 , You can include figures . These do not count)
### 1. Introduction
- The problem: classifying metagenomic reads to species and genus, and why it matters for
  identifying plant pathogens.
- The two approaches in your own words: Kraken2 (exact k-mer matching to a reference database,
  lowest common ancestor) and the fine-tuned DNA language model.
- The aim of the comparison and the question you want to answer.

### 2. Methodology
- **Reference panel:** the 18 species and the genome accessions used (a table, from Section 2).
- **Simulated reads:** simulator and error model, read length, `READS_PER_SPECIES`, and the train
  / validation / test split.
- **Kraken2:** how the database was built, how calls were translated to species and genus
  (`above_species`, `above_genus`, `unclassified`), and the `K2_CONFIDENCE` value.
- **AI model:** the model (`MODEL_NAME`), the fine-tuning strategy and settings (epochs, learning
  rate, batch size), how genus probabilities were obtained, and the `AI_MIN_CONF` value.
- **Novel species (Part D):** which species were used and why.
- **Metrics:** accuracy, precision of calls and percentage called, defined in your own words.
- **Your own experiment:** the setting you changed (see Results, point 5).

### 3. Results
You can use this steps as a reference:
1. **AI training:** the training vs validation table and the learning curves.
   *At which epoch was the validation loss lowest? Did the model learn, or memorize?*
2. **Part C, held-out reads:** the genus and species scoreboards and the confusion matrices.
   *How much does each tool lose going from genus to species? Which genera and species does the
   AI confuse, and are they related?*
3. **Speed:** inference time of each tool, and the AI's fine-tuning time.
4. **Part D, novel species:** the genus scoreboard, the percentage of false species calls, and
   the known-vs-novel bar plot. *Which method lost more accuracy at genus level? How many false
   species calls did each tool make?*
5. **Your own experiment (at least one).** Change **one** setting, re-run the cells it affects and
   report a before / after table. For example:
   - `K2_CONFIDENCE = 0.1` or `AI_MIN_CONF = 0.99`, evaluated on Part D: what changes in false
     calls and in the number of reads called?
   - more `READS_PER_SPECIES` or fewer `num_train_epochs`: does the training / validation gap
     change, and does the AI's accuracy change?

### 4. Discussion
Interpret your results:
You can use these steps as reference
1. **Learning vs memorizing.** What does the gap between training and validation accuracy tell
   you about the AI model, and what could explain it?
2. **A fair comparison?** Kraken2's database contains the complete genomes; the AI saw only the
   training reads. Estimate what fraction of each genome the AI saw. How does this affect your
   conclusions, and how would you design a fairer comparison?
3. **Not calling.** Kraken2 reports unsure reads at a higher rank or leaves them unclassified;
   the plain AI always picks a species. What are the advantages and disadvantages of "always
   guessing"? Did the confidence cut-off help? Why or why not?
4. **Novel species.** How did each tool deal with species that were not in its database or
   training data? What does that mean for species calls on a real field sample?
5. **Speed and cost.** Kraken2 needs a database; the AI needs GPU fine-tuning. Which approach
   would scale better to 1,000 species, or to adding one new genome?
6. **Limitations** of this experiment (simulated reads, panel size, one genome per species, ...).

### 5. Conclusion
A short paragraph that answers the question from the Introduction: which tool would you use to
identify the pathogen in a real plant sample, and why? Base your answer on your results.